# 2.3 Stride

The previous section described shapes, their hierarchies, and coordinates for those shapes. To construct layouts of data, threads, or other objects, we define a mapping from coordinates within a shape to offsets.

**Definition 2.15.** A stride $D$ for a shape $S$ is an HTuple($D$) that is congruent with the shape, $S \sim D$. This stride defines a mapping from a natural coordinate $\tilde{c} \in \mathbb{Z}_S$ to the codomain $D$, given by

$$\text{inner\_product}: \mathbb{Z} \cdot D \to D, \quad c \cdot d \mapsto cd$$

$$\text{inner\_product}: \text{HTuple}(\mathbb{Z}) \cdot \text{HTuple}(D) \to D, \quad c \cdot d \mapsto \sum_i \text{inner\_product}(c_i, d_i)$$

In most cases, strides are also HTuple($\mathbb{Z}$)s, meaning $D = \mathbb{Z}$. The resulting integer produced by inner\_product is typically interpreted as an offset within a data array. However, the concept of a stride element generalizes to any element of an integer-semimodule, which provides significant flexibility in the span of functions that layouts can represent.

In [1]:
from tensor_layouts import *
from tensor_layouts.viz import draw_layout

In [2]:
# crd2offset computes the inner product of a coordinate with a stride,
# giving the memory offset. For layout (4, 8) : (1, 4), coordinate (m, n):
#   offset = m * 1 + n * 4

# Example: Layout (4, 8) : (1, 4) — column-major
shape = (4, 8)
stride = (1, 4)
L = Layout(shape, stride)

# Verify inner_product matches layout evaluation for all natural coordinates
for i in range(size(L)):
    coord = idx2crd(i, shape)  # natural coordinate
    offset = crd2offset(coord, shape, stride)
    assert offset == L(i), f"Mismatch at i={i}"

# Show a few examples
for i in [0, 1, 5, 22, 31]:
    coord = idx2crd(i, shape)
    offset = crd2offset(coord, shape, stride)
    print(f"  coord {str(coord):6s} . stride {stride} = {offset:3d}  (layout({i}) = {L(i)})")


  coord (0, 0) . stride (1, 4) =   0  (layout(0) = 0)
  coord (1, 0) . stride (1, 4) =   1  (layout(1) = 1)
  coord (1, 1) . stride (1, 4) =   5  (layout(5) = 5)
  coord (2, 5) . stride (1, 4) =  22  (layout(22) = 22)
  coord (3, 7) . stride (1, 4) =  31  (layout(31) = 31)


In [3]:
# With hierarchical/nested shapes, the inner_product recurses into sub-tuples.
# Layout ((2, 2), (4, 2)) : ((1, 8), (2, 16))
# For a natural coordinate ((a, b), (c, d)):
#   offset = a*1 + b*8 + c*2 + d*16

shape = ((2, 2), (4, 2))
stride = ((1, 8), (2, 16))
L = Layout(shape, stride)

print(f"Layout: {L}")
print(f"  size = {size(L)}, cosize = {cosize(L)}")
print()

# Verify inner_product on nested coordinates
for i in range(size(L)):
    coord = idx2crd(i, shape)
    offset = crd2offset(coord, shape, stride)
    assert offset == L(i)

# Show the paper's example: integral coordinate 22
i = 22
c_2d = idx2crd(i, (4, 8))         # (2, 5) — flat 2D coordinate
c_nat = idx2crd(i, shape)   # ((0, 1), (1, 1)) — natural coordinate
offset = L(i)
print(f"L(22) = L{c_2d} = L{c_nat} = {offset}")
print()
print(f"Breakdown: {c_nat[0][0]}*1 + {c_nat[0][1]}*8 + {c_nat[1][0]}*2 + {c_nat[1][1]}*16")
print(f"         = {c_nat[0][0]*1}.  + {c_nat[0][1]*8} + {c_nat[1][0]*2} + {c_nat[1][1]*16} = {offset}")

Layout: ((2, 2), (4, 2)) : ((1, 8), (2, 16))
  size = 32, cosize = 32

L(22) = L(2, 5) = L((0, 1), (1, 1)) = 26

Breakdown: 0*1 + 1*8 + 1*2 + 1*16
         = 0.  + 8 + 2 + 16 = 26


## 2.3.1 Integer-Semimodules

**Definition 2.16.** An integer-semimodule is a set $M$ equipped with an associative addition, $M + M \to M$, and a scalar multiplication, $\mathbb{Z} \cdot M \to M$. For $a, b \in \mathbb{Z}$ and $m, n, p \in M$, the addition and scalar multiplication satisfy

- Multiplicative Identity: $1 \cdot m = m$,
- Additive Associativity: $m + (n + p) = (m + n) + p$,
- Multiplicative Associativity: $a \cdot (b \cdot m) = (ab) \cdot m$.

Additive identities and inverses are not required, so $(M, +)$ is a semigroup. We write an integer-semimodule as $(M, +, \cdot)$ to denote the set $M$, the additive operation $+$, and the scalar multiplication $\cdot$.

The integers, $\mathbb{Z}$, are an integer-semimodule.
The rationals, $\mathbb{Q}$, are an integer-semimodule.
The field $\mathbb{F}_2 = (\{0, 1\}, \text{XOR}, \text{AND})$ of arithmetic operations modulo 2 is an integer-semimodule. Any Cartesian product or HTuple of integer-semimodules is an integer-semimodule given elementwise addition and scalar multiplication.

A uniquely useful integer-semimodule is $(\mathbb{Z}_S, +, \cdot)$ with $\mathbb{Z}_S$ the set of all HTuple($\mathbb{Z}$) congruent to $S$. For instance, the basis elements of rank-2 arithmetic tuples form an integer-semimodule:

$$e_0 = (1, 0), \quad e_1 = (0, 1),$$

$$\mathbb{Z}_{(\ast,\ast)} = \{a \cdot e_0 + b \cdot e_1 \mid a, b \in \mathbb{Z}\}$$

with integer scaling and addition defined element-wise:

$$a \cdot e_0 + b \cdot e_1 = (a, b)$$

Thus, $e_0$, $e_1$, and any linear combination can be used as strides within a layout. By selecting stride elements from $\mathbb{Z}_S$, layouts can generate natural coordinates of a shape $S$ through the inner\_product operation.

In [4]:
# Demonstrate the integer-semimodule axioms with concrete examples.

# 1. The integers Z are an integer-semimodule: (Z, +, *)
m, n, p_val = 5, 3, 7
a, b = 4, 6

assert 1 * m == m                                     # Multiplicative identity
assert m + (n + p_val) == (m + n) + p_val             # Additive associativity
assert a * (b * m) == (a * b) * m                     # Multiplicative associativity
print("Z satisfies integer-semimodule axioms. ✓")

# 2. F2 = ({0, 1}, XOR, AND) is an integer-semimodule
class F2:
    """Elements of the field F2 with XOR as addition and AND as scalar multiplication."""
    def __init__(self, val):
        self.val = val & 1  # mod 2
    def __add__(self, other):
        return F2(self.val ^ other.val)   # XOR
    def __rmul__(self, scalar):            # Z * F2 -> F2
        return F2((scalar & 1) & self.val) # AND with parity of scalar
    def __eq__(self, other):
        return self.val == other.val
    def __repr__(self):
        return str(self.val)

x, y, z = F2(1), F2(0), F2(1)
assert 1 * x == x                         # Multiplicative identity
assert (x + y) + z == x + (y + z)         # Additive associativity
assert 3 * (2 * x) == (3 * 2) * x         # Multiplicative associativity
print("F2 = ({0,1}, XOR, AND) satisfies integer-semimodule axioms. ✓")
print(f"  Examples: 1 XOR 0 = {(x + y).val}, 1 XOR 1 = {(x + z).val}, 3 * F2(1) = {(3 * x).val}")

Z satisfies integer-semimodule axioms. ✓
F2 = ({0,1}, XOR, AND) satisfies integer-semimodule axioms. ✓
  Examples: 1 XOR 0 = 1, 1 XOR 1 = 0, 3 * F2(1) = 1


In [5]:
# The coordinate-tuple semimodule Z_{(*,*)} with basis e0=(1,0), e1=(0,1)
# Strides can be tuples! A layout with tuple strides generates coordinates.

# This implements the inner_product generalized to tuple-valued strides.
# When strides are tuples like (1, 0) and (0, 1), the inner_product
# uses elementwise scaling and addition to produce a coordinate tuple.

def scale(n, elem):
    """Scalar multiplication: Z * M -> M"""
    if isinstance(elem, tuple):
        return tuple(scale(n, e) for e in elem)
    return n * elem

def add(a, b):
    """Addition: M + M -> M"""
    if isinstance(a, tuple) and isinstance(b, tuple):
        return tuple(add(x, y) for x, y in zip(a, b))
    return a + b

def inner_product_general(coord, stride):
    """inner_product generalized to semimodule-valued strides."""
    if isinstance(coord, int) and not isinstance(stride, tuple):
        return scale(coord, stride)
    if isinstance(coord, int) and isinstance(stride, tuple):
        return scale(coord, stride)
    # Both are tuples — recurse and sum
    result = inner_product_general(coord[0], stride[0])
    for c, d in zip(coord[1:], stride[1:]):
        result = add(result, inner_product_general(c, d))
    return result

# Basis vectors of Z_{(*,*)}
e0 = (1, 0)
e1 = (0, 1)

# A layout with shape (3, 5) and tuple strides (e0, e1)
# maps coordinate (i, j) to: i * (1,0) + j * (0,1) = (i, j)
# This is essentially the identity layout that generates coordinates!
shape_coord = (3, 5)
stride_coord = (e0, e1)

print("Layout with tuple strides: shape (3,5), stride ((1,0), (0,1))")
print("This generates coordinates as outputs:")
print()
for j in range(5):
    for i in range(3):
        coord = idx2crd(i + j * 3, shape_coord)
        result = inner_product_general(coord, stride_coord)
        print(f"  ({i},{j}) -> {result}", end="")
    print()

Layout with tuple strides: shape (3,5), stride ((1,0), (0,1))
This generates coordinates as outputs:

  (0,0) -> (0, 0)  (1,0) -> (1, 0)  (2,0) -> (2, 0)
  (0,1) -> (0, 1)  (1,1) -> (1, 1)  (2,1) -> (2, 1)
  (0,2) -> (0, 2)  (1,2) -> (1, 2)  (2,2) -> (2, 2)
  (0,3) -> (0, 3)  (1,3) -> (1, 3)  (2,3) -> (2, 3)
  (0,4) -> (0, 4)  (1,4) -> (1, 4)  (2,4) -> (2, 4)


In [6]:
# A more interesting example: strides can be non-basis tuples.
# With stride ((2, 0), (0, 3)), coordinate (i, j) maps to (2i, 3j).
# This generates scaled coordinates — useful for strided access patterns.

stride_scaled = ((2, 0), (0, 3))

print("Layout with scaled tuple strides: shape (3,5), stride ((2,0), (0,3))")
print("Maps (i,j) -> (2i, 3j):")
print()
for j in range(5):
    row = []
    for i in range(3):
        coord = (i, j)
        result = inner_product_general(coord, stride_scaled)
        row.append(result)
    print(f"  j={j}: {row}")

Layout with scaled tuple strides: shape (3,5), stride ((2,0), (0,3))
Maps (i,j) -> (2i, 3j):

  j=0: [(0, 0), (2, 0), (4, 0)]
  j=1: [(0, 3), (2, 3), (4, 3)]
  j=2: [(0, 6), (2, 6), (4, 6)]
  j=3: [(0, 9), (2, 9), (4, 9)]
  j=4: [(0, 12), (2, 12), (4, 12)]


The key insight of integer-semimodules is that the same inner\_product mechanism used for ordinary integer strides generalizes seamlessly to other algebraic structures. By choosing strides from $\mathbb{Z}_{(\ast,\ast)}$, layouts can produce coordinates as outputs rather than integer offsets. By choosing strides from $\mathbb{F}_2$, layouts can produce XOR-based swizzle patterns.

This uniformity — the same shape/stride/inner\_product framework regardless of the codomain — is what makes CuTe layouts a powerful abstraction that goes well beyond traditional row-major/column-major indexing.